## Preparação do Ambiente e Carregamento dos Dados

### Instalação de Pacotes

In [1]:
! pip install --upgrade -q xgboost scikit-learn imbalanced-learn matplotlib optuna plotly nbformat kaleido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 65.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 81.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
bigframes 2.26.0 requi

### Importação de Bibliotecas

In [4]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
from xgboost import XGBClassifier

import optuna
import pickle
import optuna.visualization as vis

#from utils.constants import *

### Constantes

In [8]:
CATEGORICAL_COLUMNS = {
    "gestante_paciente", "raca_cor_paciente", "sigla_uf_residencia"
}

# --- Training/Testing Constants ---

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"]

TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

COARSE_LABEL_MAP = {0: 0, 1: 1, 2: 1}
FINE_LABEL_MAP = {1: 0, 2: 1}
FINE_LABEL_MAP_REVERSE = {0: 1, 1: 2}

### Carregamento dos Dados

In [33]:
dataset_path = "/kaggle/input/datasets/maximusborges/sinan-sbcas/dataset-processed-gb.csv"
df = pd.read_csv(dataset_path)

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [34]:
def get_coarse_fine_data(X, y):
    X_coarse = X.copy()
    y_coarse = y.copy()
    y_coarse = y_coarse.map(COARSE_LABEL_MAP)

    high_risk_mask = y.isin([1, 2])
    X_fine = X[high_risk_mask].copy()
    y_fine = y[high_risk_mask].copy()
    y_fine = y_fine.map(FINE_LABEL_MAP)

    return X_coarse, y_coarse, X_fine, y_fine

## Combinação dos Modelos

In [35]:
def predict_soft_cascade(model_coarse, model_fine, X, y):
    # Get Probabilities from Coarse
    probs_coarse = model_coarse.predict_proba(X)
    p_high_risk = probs_coarse[:, 1]
    
    # Get Probabilities from Fine
    probs_fine = model_fine.predict_proba(X) 
    p_severe_given_high = probs_fine[:, 1] # P(Severe | High)
        
    # P(Severe) = P(High) * P(Severe | High)
    p_severe_global = p_high_risk * p_severe_given_high
    
    # P(Alarm) = P(High) * (1 - P(Severe | High))
    p_alarm_global = p_high_risk * (1 - p_severe_given_high)
    
    # P(Low) = 1 - P(High)
    p_low_global = 1.0 - p_high_risk
    
    final_probs = np.vstack([p_low_global, p_alarm_global, p_severe_global]).T
    final_preds = np.argmax(final_probs, axis=1)

    return f1_score(y, final_preds, average='macro'), final_preds


def predict_hard_cascade(
    model_coarse, model_fine, X, y, 
    threshold_coarse=0.5, threshold_fine=0.5
):
    # Predict Coarse (0 = Low, 1 = High)
    probs_coarse = model_coarse.predict_proba(X)[:, 1]
    
    preds_coarse = (probs_coarse >= threshold_coarse).astype(int)
    final_preds = preds_coarse.copy()
    
    high_risk_indices = np.where(preds_coarse == 1)[0]
    
    if len(high_risk_indices) > 0:
        X_high_risk = X.iloc[high_risk_indices]
        
        # Predict Fine (0 = Alarm, 1 = Severe)
        probs_fine_local = model_fine.predict_proba(X_high_risk)[:, 1]
        preds_fine_local = (probs_fine_local >= threshold_fine).astype(int)
        
        # Map Fine predictions back to Global labels
        preds_fine_global = np.array([FINE_LABEL_MAP_REVERSE[p] for p in preds_fine_local])
        final_preds[high_risk_indices] = preds_fine_global

    return f1_score(y, final_preds, average='macro'), final_preds

## Treinamento e Otimização do Modelo

### Treinamento com Validação Cruzada

In [ ]:
def train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse=None, threshold_fine=None):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        # sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        clf_coarse = XGBClassifier(**params_coarse)
        clf_fine = XGBClassifier(**params_fine)

        # 1. Obtenha os dados divididos
        X_coarse, y_coarse, X_fine, y_fine = get_coarse_fine_data(X_train, y_train)
        X_coarse_valid, y_coarse_valid, X_fine_valid, y_fine_valid = get_coarse_fine_data(X_valid, y_valid)

        # 2. CALCULE OS PESOS ESPECIFICAMENTE PARA CADA CONJUNTO
        weights_coarse = compute_sample_weight(class_weight='balanced', y=y_coarse)
        weights_fine = compute_sample_weight(class_weight='balanced', y=y_fine)

        try:
            clf_coarse.fit(
                X_coarse, y_coarse,
                eval_set=[(X_coarse_valid, y_coarse_valid)], 
                sample_weight=weights_coarse,  # <-- USE PESOS COARSE
                verbose=False
            )
            
            clf_fine.fit(
                X_fine, y_fine,
                eval_set=[(X_fine_valid, y_fine_valid)], 
                sample_weight=weights_fine, # <-- USE PESOS FINE
                verbose=False
            )

            if use_soft_cascade:
                f1, _ = predict_soft_cascade(clf_coarse, clf_fine, X_valid, y_valid)
            else:
                f1, _ = predict_hard_cascade(
                    clf_coarse, clf_fine, X_valid, y_valid,
                    threshold_coarse, threshold_fine
                )
            
            scores.append(f1)
            
        except ValueError as e:
            print(f"ERRO NO TREINAMENTO: {e}")
            return 0.0, 1.0
        
    return np.mean(scores), np.std(scores)

### Definição da Função Objetivo

In [37]:
FIXED_PARAMS = {
    #"n_estimators": 2000,
    "early_stopping_rounds": 10,
    "enable_categorical": True,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "mlogloss",
    "random_state": RANDOM_STATE,
    "verbosity": 0
}

In [43]:
def objective(trial: optuna.trial.Trial):
    params_coarse = {
        "n_estimators": trial.suggest_int("n_estimators_coarse", 1500, 2000, step=100),
        "max_depth": trial.suggest_int("max_depth_coarse", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate_coarse", 1e-3, 1e-1, log=True),
        "subsample": trial.suggest_float("subsample_coarse", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree_coarse", 0.1, 1.0, step=0.1),
        "reg_alpha": trial.suggest_int("reg_alpha_coarse", 1, 50),
        "reg_lambda": trial.suggest_int("reg_lambda_coarse", 1, 50),
        "min_child_weight": trial.suggest_int("min_child_weight_coarse", 1, 10),
        "gamma": trial.suggest_float("gamma_coarse", 0, 5, step=0.5),
        **FIXED_PARAMS
    }
    
    params_fine = {
        "n_estimators": trial.suggest_int("n_estimators_fine", 2000, 3000, step=100),
        "max_depth": trial.suggest_int("max_depth_fine", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate_fine", 1e-3, 1e-1, log=True),
        "subsample": trial.suggest_float("subsample_fine", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree_fine", 0.1, 1.0, step=0.1),
        "reg_alpha": trial.suggest_int("reg_alpha_fine", 1, 50),
        "reg_lambda": trial.suggest_int("reg_lambda_fine", 1, 50),
        "min_child_weight": trial.suggest_int("min_child_weight_fine", 1, 10),
        "gamma": trial.suggest_float("gamma_fine_fine", 0, 5, step=0.5),
        **FIXED_PARAMS
    }

    mode = trial.suggest_categorical("mode", choices=["soft", "hard"])

    threshold_coarse = None
    threshold_fine = None
    use_soft_cascade = (mode == "soft")
    
    if not use_soft_cascade:
        threshold_coarse = trial.suggest_float("threshold_coarse", low=0.25, high=0.55)
        threshold_fine = trial.suggest_float("threshold_fine", low=0.25, high=0.55)

    avg, _ = train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse, threshold_fine)

    return avg

### Otimização com Optuna

In [44]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

[I 2026-02-11 02:07:20,931] A new study created in memory with name: no-name-6bc87439-d93f-42a2-890f-e886023ef124


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-11 02:09:18,720] Trial 0 finished with value: 0.46325391519694514 and parameters: {'n_estimators_coarse': 1700, 'max_depth_coarse': 3, 'learning_rate_coarse': 0.001243477378105289, 'subsample_coarse': 0.6, 'colsample_bytree_coarse': 0.7000000000000001, 'reg_alpha_coarse': 38, 'reg_lambda_coarse': 5, 'min_child_weight_coarse': 8, 'gamma_coarse': 0.5, 'n_estimators_fine': 2000, 'max_depth_fine': 12, 'learning_rate_fine': 0.008360903753410334, 'subsample_fine': 0.5, 'colsample_bytree_fine': 0.9, 'reg_alpha_fine': 18, 'reg_lambda_fine': 28, 'min_child_weight_fine': 7, 'gamma_fine_fine': 4.0, 'mode': 'soft'}. Best is trial 0 with value: 0.46325391519694514.
[I 2026-02-11 02:11:08,157] Trial 1 finished with value: 0.4943398338653247 and parameters: {'n_estimators_coarse': 1900, 'max_depth_coarse': 2, 'learning_rate_coarse': 0.07035700409594293, 'subsample_coarse': 0.9, 'colsample_bytree_coarse': 0.9, 'reg_alpha_coarse': 25, 'reg_lambda_coarse': 28, 'min_child_weight_coarse': 4, 'g

In [46]:
output_dir = Path('results/optimization/xgboost/hier')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [58]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

vis.plot_param_importances(study).write_image(output_dir / 'opt_param_importances.png')
vis.plot_optimization_history(study).write_image(output_dir / 'opt_hist.png')

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


### Melhor Modelo

Best F1: 0.513032606189061  

Best Params: {'n_estimators_coarse': 1600, 'max_depth_coarse': 14, 'learning_rate_coarse': 0.06310200379301271, 'subsample_coarse': 0.8, 'colsample_bytree_coarse': 0.5, 'reg_alpha_coarse': 5, 'reg_lambda_coarse': 47, 'min_child_weight_coarse': 9, 'gamma_coarse': 0.0, 'n_estimators_fine': 2600, 'max_depth_fine': 2, 'learning_rate_fine': 0.0015059118006550702, 'subsample_fine': 0.8, 'colsample_bytree_fine': 0.7000000000000001, 'reg_alpha_fine': 38, 'reg_lambda_fine': 23, 'min_child_weight_fine': 7, 'gamma_fine_fine': 4.5, 'mode': 'soft'}

In [51]:
# 1. Separar os parâmetros baseados nos sufixos '_coarse' e '_fine'
best_params = best_trial.params

params_coarse = {**FIXED_PARAMS}
params_fine = {**FIXED_PARAMS}

for key, value in best_params.items():
    if key.endswith('_coarse'):
        # Remove o sufixo '_coarse' para o nome do parametro real do XGBoost
        real_key = key.replace('_coarse', '')
        params_coarse[real_key] = value
    elif key.endswith('_fine'):
        # Remove o sufixo '_fine'
        real_key = key.replace('_fine', '')
        params_fine[real_key] = value

# 2. Configurar o modo de cascata e thresholds
mode = best_params.get('mode', 'soft')
use_soft_cascade = (mode == 'soft')

threshold_coarse = best_params.get('threshold_coarse', None)
threshold_fine = best_params.get('threshold_fine', None)

In [52]:
# 3. Chamar a função de treino com os argumentos separados
avg_f1_final, std_f1_final = train_on_folds(
    params_coarse, 
    params_fine, 
    use_soft_cascade, 
    threshold_coarse, 
    threshold_fine
)

print(f"Média F1: {avg_f1_final}")
print(f"Desvio Padrão F1: {std_f1_final}")

Média F1: 0.513032606189061
Desvio Padrão F1: 0.0018415995997279543
